In [22]:
# Dataset Load
import seaborn as sns
df=sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [24]:
df['sex'].value_counts()

sex
Male      157
Female     87
Name: count, dtype: int64

In [25]:
df['smoker'].value_counts()

smoker
No     151
Yes     93
Name: count, dtype: int64

In [26]:
df['day'].value_counts()

day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

In [27]:
df['time'].value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

### Feature Encoding (Label Encoding and OneHot Encoding)

### independent and dependent features >>>>>

In [28]:
df.columns

Index(['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size'], dtype='str')

In [29]:
## independnent and dependent features
X=df[['tip', 'sex', 'smoker', 'day', 'time', 'size']]
y=df['total_bill']

In [30]:
##train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=10)

In [31]:
X_train.head()

,tip,sex,smoker,day,time,size
58,1.76,Male,Yes,Sat,Dinner,2
1,1.66,Male,No,Sun,Dinner,3
2,3.50,Male,No,Sun,Dinner,3
68,2.01,Male,No,Sat,Dinner,2
184,3.00,Male,Yes,Sun,Dinner,2


## Feature Encoding(LAbel Encoding And Onehot Encoding)

In [32]:
from sklearn.preprocessing import LabelEncoder

le1=LabelEncoder()
le2=LabelEncoder()
le3=LabelEncoder()

In [33]:
import warnings
warnings.filterwarnings('ignore')

X_train['sex']=le1.fit_transform(X_train['sex'])
X_train['smoker']=le2.fit_transform(X_train['smoker'])
X_train['time']=le3.fit_transform(X_train['time'])

In [34]:
X_train.head()

,tip,sex,smoker,day,time,size
58,1.76,1,1,Sat,0,2
1,1.66,1,0,Sun,0,3
2,3.50,1,0,Sun,0,3
68,2.01,1,0,Sat,0,2
184,3.00,1,1,Sun,0,2


In [35]:
X_test['sex']=le1.transform(X_test['sex'])
X_test['smoker']=le2.transform(X_test['smoker'])
X_test['time']=le3.transform(X_test['time'])

In [36]:
X_test.head()

,tip,sex,smoker,day,time,size
162,2.00,0,0,Sun,0,3
60,3.21,1,1,Sat,0,2
61,2.00,1,1,Sat,0,2
63,3.76,1,1,Sat,0,4
69,2.09,1,1,Sat,0,2


### Onehot encoding with >>>>>>  ColumnTrnasformer

In [37]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [43]:
ct=ColumnTransformer(transformers=[('onehot',OneHotEncoder(drop='first'),[3])],
                                   remainder='passthrough') # namer jagay jekono nam.

In [45]:
X_train = ct.fit_transform(X_train)

In [55]:
X_train

array([[1., 0., 0., ..., 1., 0., 2.],
       [0., 1., 0., ..., 0., 0., 3.],
       [0., 1., 0., ..., 0., 0., 3.],
       ...,
       [1., 0., 0., ..., 0., 0., 2.],
       [0., 0., 1., ..., 0., 1., 6.],
       [0., 1., 0., ..., 0., 0., 2.]], shape=(183, 8))

In [56]:
import sys
import numpy
numpy.set_printoptions(threshold=sys.maxsize)

In [57]:
X_test=ct.transform(X_test)

### SVR--Support Vector Regression

In [59]:
from sklearn.svm import SVR
svr=SVR()

In [60]:
svr.fit(X_train,y_train)

SVR()

In [61]:
y_pred=svr.predict(X_test)

In [63]:
from sklearn.metrics import r2_score,mean_absolute_error

print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.4602811456115927
4.1486423210190235


### Hyperparameter Tuning using GridSearch CV

In [64]:
from sklearn.model_selection import GridSearchCV
 
# defining parameter range
param_grid = {'C': [0.1, 1, 10, 100, 1000],
               'gamma': [1, 0.1, 0.01, 0.001, 0.0001, 'scale', 'auto'], # Kernel coefficient
                'kernel': ['rbf', 'linear', 'poly', 'sigmoid'] # Different kernel types
             }

In [65]:
grid = GridSearchCV(SVR(), param_grid, refit = True, verbose = 3, n_jobs=-1)
 
# fitting the model with the grid search
grid.fit(X_train, y_train)

Fitting 5 folds for each of 140 candidates, totalling 700 fits


GridSearchCV(estimator=SVR(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': [1, 0.1, 0.01, 0.001, 0.0001, 'scale',
                                   'auto'],
                         'kernel': ['rbf', 'linear', 'poly', 'sigmoid']},
             verbose=3)

In [66]:
grid.best_params_

{'C': 100, 'gamma': 1, 'kernel': 'linear'}

In [67]:
grid_predict = grid.predict(X_test)

In [68]:
print(r2_score(y_test,grid_predict))
print(mean_absolute_error(y_test,grid_predict))

0.5453723337106238
3.912971558542979
